# 图形 API 概念
## 图表¶
LangGraph 的核心是将代理工作流程建模为图表。您可以使用三个关键组件来定义代理的行为：

- [State](https://langchain-ai.github.io/langgraph/concepts/low_level/#state)：表示应用程序当前快照的共享数据结构。它可以是任何 Python 类型，但通常是 aTypedDict或 Pydantic BaseModel。

- [Nodes](https://langchain-ai.github.io/langgraph/concepts/low_level/#nodes)：用于编码代理逻辑的 Python 函数。它们接收当前值State作为输入，执行一些计算或副作用，并返回更新后的State。

- [EdgesNode](https://langchain-ai.github.io/langgraph/concepts/low_level/#edges)：根据当前条件确定下一步执行哪个操作的 Python 函数State。它们可以是条件分支或固定转换。


通过组合Nodes和Edges，您可以创建复杂的循环工作流，使其State随时间推移而演化。然而，真正的强大之处在于 LangGraph 对 的管理方式State。需要强调的是：Nodes和Edges只不过是 Python 函数而已——它们可以包含 LLM 代码，也可以只是经典的 Python 代码。

简而言之：节点完成工作，边告诉下一步做什么。

LangGraph 的底层图算法使用消息传递来定义通用程序。当一个节点完成其操作时，它会沿着一条或多条边向其他节点发送消息。这些接收节点随后执行其函数，将生成的消息传递给下一组节点，并继续执行该过程。受 Google Pregel系统的启发，该程序以离散的“超级步骤”进行。

超级步骤可以被视为图节点上的单次迭代。并行运行的节点属于同一个超级步骤，而顺序运行的节点则属于不同的超级步骤。图执行开始时，所有节点都处于同一inactive状态。当节点active在其任何传入边（或“通道”）上收到新消息（状态）时，它将变为 。然后，活动节点运行其函数并进行更新响应。在每个超级步骤结束时，没有传入消息的节点halt通过将自己标记为来投票 。当所有节点 均为inactive且没有消息在传输时，inactive图执行终止。


## 状态图¶
该类`StateGraph`是要使用的主要图形类。它由用户定义的State对象参数化。

## 编译你的图表¶
要构建图，首先要定义状态，然后添加节点和边，最后进行编译。编译图究竟是什么？为什么需要编译？

编译是一个非常简单的步骤。它会对图的结构进行一些基本检查（例如，没有孤立节点等）。你还可以在其中指定运行时参数，例如检查点和断点。只需调用以下.compile方法即可编译图：

```python
graph = graph_builder.compile(...)
```

您必须先编译您的图表，然后才能使用它。

### 状态
定义图形的第一件事就是定义图形的状态。状态由图的模式和指定如何对状态应用更新的还原函数组成。状态的模式将是图中所有节点和边的输入模式，可以是 TypedDict 或 Pydantic 模型。所有节点都将向状态发送更新，然后使用指定的还原器函数应用这些更新。

### 架构¶
指定图形模式的主要记录方式是使用TypedDict。如果您想在状态中提供默认值，请使用dataclass。如果您需要递归数据验证，我们也支持使用 Pydantic BaseModel作为图形状态（但请注意，pydantic 的性能不如TypedDict或dataclass）。

默认情况下，图将具有相同的输入和输出模式。如果您想更改此设置，也可以直接指定显式的输入和输出模式。当您拥有大量键，并且其中一些显式用于输入，另一些用于输出时，这非常有用。请参阅此处的指南以了解如何使用。

### 多个模式¶
通常，所有图节点都使用同一个模式进行通信。这意味着它们将读取和写入相同的状态通道。但是，在某些情况下，我们希望对此有更多控制：

内部节点可以传递图的输入/输出中不需要的信息。
我们可能还想为图使用不同的输入/输出模式。例如，输出可能只包含一个相关的输出键。
可以让节点写入图内的私有状态通道，以实现节点内部通信。我们可以简单地定义一个私有模式。更多详细信息，PrivateState请参阅本指南。

也可以为图定义显式的输入和输出模式。在这种情况下，我们会定义一个“内部”模式，其中包含与图操作相关的所有input键。但是，我们还会定义“内部”模式的子集和output模式，以约束图的输入和输出。更多详情，请参阅本指南。

让我们看一个例子：




In [ ]:
class InputState(TypedDict):
    user_input: str

class OutputState(TypedDict):
    graph_output: str

class OverallState(TypedDict):
    foo: str
    user_input: str
    graph_output: str

class PrivateState(TypedDict):
    bar: str

def node_1(state: InputState) -> OverallState:
    # Write to OverallState
    return {"foo": state["user_input"] + " name"}

def node_2(state: OverallState) -> PrivateState:
    # Read from OverallState, write to PrivateState
    return {"bar": state["foo"] + " is"}

def node_3(state: PrivateState) -> OutputState:
    # Read from PrivateState, write to OutputState
    return {"graph_output": state["bar"] + " Lance"}

builder = StateGraph(OverallState,input_schema=InputState,output_schema=OutputState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
graph.invoke({"user_input":"My"})
{'graph_output': 'My name is Lance'}

这里有两个微妙而重要的问题需要注意：

- 我们将 `state：InputState` 作为 `node_1` 的输入模式。但是，我们写入的是 `OverallState` 中的一个通道 `foo`。我们怎么能写入输入模式中不包含的状态通道呢？这是因为节点可以写入图状态中的任何状态通道。图状态是初始化时定义的状态通道的组合，其中包括 `OverallState` 以及过滤器 `InputState` 和 `OutputState`。
- `StateGraph(OverallState,input_schema=InputState,output_schema=OutputState)` 对图形进行初始化。那么，我们如何在`node_2` 中写入 `PrivateState`？如果在 `StateGraph` 初始化时没有传递该模式，那么图如何访问该模式呢？我们可以这样做，因为只要状态模式定义存在，节点也可以声明额外的状态通道。在本例中，`PrivateState` 模式已定义，因此我们可以在图中添加 `bar` 作为新的状态通道，并向其写入内容。

### Reducers¶
`Reducer` 是理解节点更新如何应用于 的关键`State`。 中的每个键`State`都有其独立的 `Reducer` 函数。如果没有明确指定 `Reducer` 函数，则假定对该键的所有更新都应覆盖该函数。`Reducer` 有几种不同的类型，首先是默认类型的 `Reducer`：

### 默认 Reducer¶
这两个示例展示了如何使用默认的 `reducer`：



In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    foo: int
    bar: list[str]

在此示例中，未为任何键指定任何 Reducer 函数。假设图的输入为{"foo": 1, "bar": ["hi"]}。然后假设第一个节点Node返回{"foo": 2}。这被视为对状态的更新。请注意，Node不需要返回整个State架构 - 只需返回更新即可。应用此更新后，State将是{"foo": 2, "bar": ["hi"]}。如果第二个节点返回{"bar": ["bye"]}，State则 将是{"foo": 2, "bar": ["bye"]}

示例 B：

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    foo: int
    bar: Annotated[list[str], add]

在这个示例中，我们使用`Annotated`为第二个键（`bar`）指定了一个`reducer`函数（`operator.add`）。请注意，第一个键保持不变。假设图的输入是 `{"foo"：1，"bar"：["hi"]}`。然后假设第一个节点返回 `{"foo"：2}`.这将被视为对状态的更新。请注意，Node 不需要返回整个状态模式，只需返回一个更新。应用此更新后，状态将是 `{"foo"：2, "bar"：["hi"]}`。如果第二个节点返回 `{"bar"：["bye"]}` 则状态为 `{"foo"：2, "bar"：["hi", "bye"]}`。请注意，这里的 `"bar "`键是通过将两个列表相加来更新的。

## 使用图形状态中的消息¶
### 为什么要使用消息？¶

大多数现代 LLM 提供商都提供聊天模型接口，接受消息列表作为输入。`LangChainChatModel`尤其接受对象列表`Message`作为输入。这些消息有多种形式，例如`HumanMessage`（用户输入）或`AIMessage`（LLM 响应）。要了解更多关于消息对象的概念，请参阅此概念指南。

### 在图表中使用消息
在许多情况下，将以前的对话历史记录存储为图形状态中的消息列表会很有帮助。为此，我们可以在存储 `Message` 对象列表的图形状态中添加一个键（通道），并使用 `reducer` 函数对其进行注释（请参阅下面示例中的 `messages` 键）。`reducer` 函数对于告诉图形如何在每次状态更新时更新状态中的 `Message` 对象列表至关重要（例如，当节点发送更新时）。如果未指定 `reducer`，则每次状态更新都会使用最近提供的值覆盖消息列表。如果只想将消息附加到现有列表中，可以使用 `operator.add` 作为 `reducer`。

不过，您也可能希望手动更新图状态中的消息（例如，"人在回路中"）。如果使用 `operator.add`，发送到图中的手动状态更新将被附加到现有的消息列表中，而不是更新现有消息。为了避免这种情况，您需要一个能跟踪消息 ID 并在更新时覆盖现有消息的还原器。为此，可以使用预建的 `add_messages` 函数。对于全新的消息，它会简单地追加到现有列表中，但也会正确处理现有消息的更新。

### 序列化
除了跟踪消息 ID 之外，每当通道收到状态更新时，该add_messages函数还会尝试将消息反序列化为 LangChain对象。有关 LangChain 序列化/反序列化的更多信息，请参阅此处。这允许以以下格式发送图输入/状态更新：`Messagemessages`





In [ ]:
# this is supported
{"messages": [HumanMessage(content="message")]}

# and this is also supported
{"messages": [{"type": "human", "content": "message"}]}

由于在使用 `add_messages` 时，状态更新总是被反序列化为 `LangChain` 消息，因此应使用点符号来访问消息属性，如 `state["消息"][-1].content`。下面是一个使用 `add_messages` 作为`reducer`函数的图形示例。

API Reference: [AnyMessage](https://python.langchain.com/api_reference/core/messages/langchain_core.messages.AnyMessage.html?_gl=1*1ujbtlt*_gcl_au*NDE1NjY4Mjc0LjE3NTM0Mjc3MTc.*_ga*MTEyMjY1OTA5MS4xNzUzNDI3NzE4*_ga_47WX3HKKY2*czE3NTM1OTUzNTckbzMkZzAkdDE3NTM1OTUzNTckajYwJGwwJGgw) | [add_messages](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.message.add_messages)



In [ ]:
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict

class GraphState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

`MessagesState` 的预构建状态，可以轻松使用消息。`MessagesState` 使用单个消息键（即 `AnyMessage` 对象列表）定义，并使用 `add_messages` 还原器。通常情况下，需要跟踪的状态不仅仅是消息，所以我们会看到有人对这个状态进行子类化，并添加更多的字段，例如

In [ ]:
from langgraph.graph import MessagesState

class State(MessagesState):
    documents: list[str]

### 节点¶
在 LangGraph 中，节点是接受以下参数的 Python 函数（同步或异步）：

- `state`：图表的[状态](https://langchain-ai.github.io/langgraph/concepts/low_level/#state)
- `config`：RunnableConfig包含配置信息（例如）thread_id和跟踪信息（例如`tags`
- `runtime`：Runtime包含运行时[context](https://langchain-ai.github.io/langgraph/concepts/low_level/#runtime-context)和其他信息的对象store，例如stream_writer
与类似，您可以使用[`add_nodeNetworkX`](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.state.StateGraph.add_node)方法将这些节点添加到图中：

API 参考：[RunnableConfig](https://python.langchain.com/api_reference/core/runnables/langchain_core.runnables.config.RunnableConfig.html?_gl=1*16pk9av*_gcl_au*NDE1NjY4Mjc0LjE3NTM0Mjc3MTc.*_ga*MTEyMjY1OTA5MS4xNzUzNDI3NzE4*_ga_47WX3HKKY2*czE3NTM1OTUzNTckbzMkZzAkdDE3NTM1OTUzNTckajYwJGwwJGgw) | [StateGraph](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.state.StateGraph)

In [ ]:
from dataclasses import dataclass
from typing_extensions import TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph
from langgraph.runtime import Runtime

class State(TypedDict):
    input: str
    results: str

@dataclass
class Context:
    user_id: str

builder = StateGraph(State)

def plain_node(state: State):
    return state

def node_with_runtime(state: State, runtime: Runtime[Context]):
    print("In node: ", runtime.context.user_id)
    return {"results": f"Hello, {state['input']}!"}

def node_with_config(state: State, config: RunnableConfig):
    print("In node with thread_id: ", config["configurable"]["thread_id"])
    return {"results": f"Hello, {state['input']}!"}


builder.add_node("plain_node", plain_node)
builder.add_node("node_with_runtime", node_with_runtime)
builder.add_node("node_with_config", node_with_config)
...

在后台，函数被转换为`RunnableLambda`，它为您的函数添加批处理和异步支持，以及本机跟踪和调试。

如果向图中添加节点而不指定名称，则会为其赋予与函数名称等效的默认名称。

### START节点¶
`NodeSTART`是一个特殊节点，表示将用户输入发送到图的节点。引用此节点的主要目的是确定应首先调用哪些节点。

API 参考：[START](https://langchain-ai.github.io/langgraph/reference/constants/#langgraph.constants.START)

In [ ]:
from langgraph.graph import START

graph.add_edge(START, "node_a")

### END节点¶
`NodeEND`是一个特殊节点，表示终端节点。当需要指示哪些边在完成后没有操作时，可以引用此节点。

In [ ]:
from langgraph.graph import END

graph.add_edge("node_a", END)

节点缓存¶
LangGraph 支持根据节点的输入缓存任务/节点。使用缓存的方法如下：

- 编译图表时指定缓存（或指定入口点）
- 指定节点的缓存策略。每个缓存策略支持：
    - `key_func`用于根据节点的输入生成缓存键，默认为`hash`使用 `pickle` 的输入。
    - `ttl`，缓存的生存时间（以秒为单位）。如果未指定，则缓存永不过期。
例如：

API 参考：[StateGraph](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.state.StateGraph)

In [ ]:
import time
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.cache.memory import InMemoryCache
from langgraph.types import CachePolicy


class State(TypedDict):
    x: int
    result: int


builder = StateGraph(State)


def expensive_node(state: State) -> dict[str, int]:
    # expensive computation
    time.sleep(2)
    return {"result": state["x"] * 2}


builder.add_node("expensive_node", expensive_node, cache_policy=CachePolicy(ttl=3))
builder.set_entry_point("expensive_node")
builder.set_finish_point("expensive_node")

graph = builder.compile(cache=InMemoryCache())

print(graph.invoke({"x": 5}, stream_mode='updates'))  
[{'expensive_node': {'result': 10}}]
print(graph.invoke({"x": 5}, stream_mode='updates'))  
[{'expensive_node': {'result': 10}, '__metadata__': {'cached': True}}]

### 边缘¶
边定义了逻辑的路由方式以及图的终止方式。这在很大程度上决定了代理的工作方式以及不同节点之间的通信方式。边有几种主要类型：

- 普通边：直接从一个节点到下一个节点。
- 条件边：调用一个函数来确定下一步要去哪个节点。
- 入口点：当用户输入到达时首先调用哪个节点。
- 条件入口点：调用一个函数来确定当用户输入到达时首先调用哪个节点。

一个节点可以有多个出边。如果一个节点有多个出边，则所有这些目标节点将作为下一个超级步骤的一部分并行执行。

法线边¶
如果总是想从节点A到节点B，那么可以直接使用[add_edge](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.state.StateGraph.add_edge)方法。

In [ ]:
graph.add_edge("node_a", "node_b")

### 条件边¶
如果您希望选择性地路由到一条或多条边（或选择性地终止），可以使用[add_conditional_edges](https://langchain-ai.github.io/langgraph/reference/graphs/#langgraph.graph.state.StateGraph.add_conditional_edges)方法。此方法接受节点名称以及执行该节点后要调用的“路由函数”。

In [ ]:
graph.add_conditional_edges("node_a", routing_function)

与节点类似，接受图的`routing_function`当前`state`并返回一个值。

默认情况下，返回值`routing_function`将用作将状态发送到下一个节点（或节点列表）的名称。所有这些节点将作为下一个超级步骤的一部分并行运行。

您可以选择提供一个字典，将 `routing_function`的输出映射到下一个节点的名称。

In [ ]:
graph.add_conditional_edges("node_a", routing_function, {True: "node_b", False: "node_c"})

> Command如果您想在单个函数中结合状态更新和路由，请使用而不是条件边。

### 入口点¶
入口点是图启动时运行的第一个节点。您可以使用`add_edge`从虚拟`START`节点到第一个要执行的节点的方法来指定图的入口点。

API 参考：[START](https://langchain-ai.github.io/langgraph/reference/constants/#langgraph.constants.START)

In [ ]:
from langgraph.graph import START

graph.add_edge(START, "node_a")

### 条件入口点¶
条件入口点允许您根据自定义逻辑从不同的节点启动。您可以使用add_conditional_edges虚拟START节点来实现这一点。

API 参考：[START](https://langchain-ai.github.io/langgraph/reference/constants/#langgraph.constants.START)



In [ ]:
from langgraph.graph import START

graph.add_conditional_edges(START, routing_function)

您可以选择提供一个字典，将 `routing_function`的输出映射到下一个节点的名称。

In [ ]:
graph.add_conditional_edges(START, routing_function, {True: "node_b", False: "node_c"})

### Send¶
默认情况下，`Nodes`和`Edges`是提前定义的，并在相同的共享状态下运行。但是，在某些情况下，确切的边无法提前知道，并且/或者您可能希望`State`同时存在 的不同版本。一个常见的例子是`map-reduce`设计模式。在这种设计模式中，第一个节点可能会生成一个对象列表，您可能希望将其他节点应用于所有这些对象。对象的数量可能提前未知（这意味着边的数量可能未知），并且`State`下游的输入`Node`应该不同（每个生成的对象对应一个输入）。

为了支持这种设计模式，LangGraph 支持`Send`从条件边返回对象。`Send`接受两个参数：第一个是节点的名称，第二个是传递给该节点的状态。

In [ ]:
def continue_to_jokes(state: OverallState):
    return [Send("generate_joke", {"subject": s}) for s in state['subjects']]

graph.add_conditional_edges("node_a", continue_to_jokes)

### Command¶
将控制流（边）和状态更新（节点）结合起来会很有用。例如，你可能希望在同一个节点中同时执行状态更新和决定下一步要转到哪个节点。LangGraph 提供了一种方法，它通过[Command](https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.Command)从节点函数返回一个对象来实现：

In [ ]:
def my_node(state: State) -> Command[Literal["my_other_node"]]:
    return Command(
        # state update
        update={"foo": "bar"},
        # control flow
        goto="my_other_node"
    )

您Command还可以实现动态控制流行为（与[条件边](https://langchain-ai.github.io/langgraph/concepts/low_level/#conditional-edges)相同）：

In [ ]:
def my_node(state: State) -> Command[Literal["my_other_node"]]:
    if state["foo"] == "bar":
        return Command(update={"foo": "baz"}, goto="my_other_node")

> 在节点函数中返回时Command，必须添加返回类型注释，其中包含节点路由到的节点名称列表，例如`Command[Literal["my_other_node"]]`。这对于图形渲染是必需的，它告诉 `LangGraphmy_node`可以导航到`my_other_node`。

请查看本操作指南，了解如何使用的端到端示例Command。

### 什么时候应该使用命令而不是条件边？¶
Command当您需要同时更新图形状态和路由到其他节点时使用。例如，在实现多代理切换时，需要路由到其他代理并向该代理传递一些信息。

使用条件边在节点之间有条件地路由而不更新状态。

## 什么时候应该使用命令而不是条件边？¶
Command当您需要同时更新图形状态和路由到其他节点时使用。例如，在实现多代理切换时，需要路由到其他代理并向该代理传递一些信息。

使用条件边在节点之间有条件地路由而不更新状态。

### 导航到父图中的节点¶
如果您正在使用[子图](https://langchain-ai.github.io/langgraph/concepts/subgraphs/)，您可能希望从子图中的一个节点导航到另一个子图（即父图中的另一个节点）。为此，您可以`graph=Command.PARENT`在 中指定Command：

In [ ]:
def my_node(state: State) -> Command[Literal["other_subgraph"]]:
    return Command(
        update={"foo": "bar"},
        goto="other_subgraph",  # where `other_subgraph` is a node in the parent graph
        graph=Command.PARENT
    )

> 设置graph为Command.PARENT将导航到最近的父图。

> 当您将某个键的更新从子图节点发送到父图节点时，如果该键由父图状态模式和子图状态模式共享，则必须为父图状态中要更新的键定义一个Reducer 。[请参阅此示例](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers)。

### 人机交互¶
Command是人机交互工作流程的重要组成部分：当使用interrupt()收集用户输入时，Command会使用 提供输入并通过 恢复执行Command(resume="User input")。查看此概念指南了解更多信息。

## 图形迁移¶
即使使用检查点来跟踪状态，LangGraph 也可以轻松处理图形定义（节点、边和状态）的迁移。

- 对于图末尾的线程（即未中断），您可以更改图的整个拓扑（即所有节点和边，删除、添加、重命名等）
- 对于当前中断的线程，我们支持除重命名/删除节点之外的所有拓扑更改（因为该线程现在可能即将进入不再存在的节点） - 如果这是一个阻止程序，请联系我们，我们可以优先解决。
- 对于修改状态，我们对添加和删除键具有完全的向后和向前兼容性
- 重命名的状态键会丢失其在现有线程中保存的状态
- 状态键的类型以不兼容的方式发生变化目前可能会导致具有更改之前状态的线程出现问题 - 如果这是一个阻碍，请联系我们，我们可以优先解决。

### 运行时上下文¶
创建图时，您可以指定传递`context_schema`给节点的运行时上下文。这对于将不属于图状态的信息传递给节点非常有用。例如，您可能希望传递模型名称或数据库连接等依赖项。

In [ ]:
@dataclass
class ContextSchema:
    llm_provider: str = "openai"

graph = StateGraph(State, context_schema=ContextSchema)

然后，您可以使用 `invoke` 方法的 `context` 参数将此上下文传递到图形中。

In [ ]:
graph.invoke(inputs, context={"llm_provider": "anthropic"})

然后，您就可以在节点或条件边中访问和使用该上下文：

In [ ]:
from langgraph.runtime import Runtime

def node_a(state: State, runtime: Runtime[ContextSchema]):
    llm = get_llm(runtime.context.llm_provider)
    ...

有关配置的详细介绍，请参阅本[指南](https://langchain-ai.github.io/langgraph/how-tos/graph-api/#add-runtime-configuration)。

## 递归限制¶
递归限制设置了图在单次执行期间可以执行的超级步骤的最大数量。一旦达到限制，LangGraph 将提升`GraphRecursionError`。默认情况下，此值设置为 25 步。可以在运行时在任何图上设置递归限制，并将其传递给`.invoke/.stream`通过配置字典。重要的是，`recursion_limit`是一个独立的config键，不应`configurable`像所有其他用户定义的配置一样在键内传递。请参见以下示例：



In [ ]:
graph.invoke(inputs, config={"recursion_limit": 5}, context={"llm": "anthropic"})

阅读本指南以了解有关递归限制如何工作的更多信息。

## 可视化¶
能够将图形可视化通常是一件很棒的事情，尤其是在图形变得越来越复杂的时候。LangGraph 内置了几种可视化图形的方法。更多信息，请参阅此[操作指南](https://langchain-ai.github.io/langgraph/how-tos/graph-api/#visualize-your-graph)。